# Deploy Qwen3.5-9B on SageMaker with Custom Model Parser for Strands Agent

### What You'll Learn

By the end of this notebook, you will understand how to:

1. **Deploy Qwen3.5-9B to SageMaker** - Create a real-time inference endpoint using the vLLM DLC container
2. **Test the endpoint** - Verify the deployment with text and streaming inference
3. **Create Strands agents** - Build AI agents that use your deployed Qwen3.5-9B model

### Model Overview

[Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B) is a 9B parameter multimodal model from the Qwen3.5 family featuring:

- **Hybrid Architecture**: Gated Delta Networks combined with Gated Attention for efficient inference
- **Thinking Mode**: Built-in chain-of-thought reasoning via `<think>` blocks
- **Tool Calling**: Native support for function calling with `qwen3_coder` parser
- **262K Context**: Native context length of 262,144 tokens
- **Multi-Token Prediction (MTP)**: Trained with MTP for speculative decoding support

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3 strands-agents strands-agents-tools mypy-boto3-sagemaker-runtime python-dotenv

In [ ]:
%load_ext autoreload
%autoreload 2

## Section 1: Environment Setup

We define the AWS region **once** here. All subsequent cells reference this single `REGION` variable.

The `model_name` is persisted to a `.env` file so other notebooks in this project can reference the active endpoint.

In [ ]:
import os
import re
import json
import time
import boto3
from pathlib import Path
from dotenv import load_dotenv, set_key
from IPython.display import clear_output

# ============================================================
# SINGLE REGION DEFINITION — referenced by all cells below
# ============================================================
REGION = "ap-south-1" #define the AWS region you want to use

# Environment file for cross-notebook variable sharing
ENV_FILE = Path(".") / ".env"

# Load existing .env if present
load_dotenv(ENV_FILE)

# Persist region to .env for cross-notebook reference
if not ENV_FILE.exists():
    ENV_FILE.touch()
set_key(str(ENV_FILE), "AWS_REGION", REGION)

# Initialize boto3 clients using the single region variable
boto_session = boto3.Session(region_name=REGION)
sm = boto_session.client("sagemaker")
sm_runtime = boto_session.client("sagemaker-runtime")
iam = boto_session.client("iam")
sts = boto_session.client("sts")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    """Wait for SageMaker endpoint to become InService."""
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)

    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]

    while status == "Creating":
        time.sleep(sleep_time)
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        clear_output(wait=True)
        progress += ind
        print(progress)

    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


print("=" * 60)
print("Environment Configuration")
print("=" * 60)
print(f"Region:    {REGION}")
print(f"Env File:  {ENV_FILE.resolve()}")
print("=" * 60)

## Section 2: Create IAM Role for SageMaker Endpoint Deployment

This section creates a dedicated IAM role with the minimum permissions needed to:
- Create/delete SageMaker models, endpoint configs, and endpoints
- Invoke endpoints (both standard and streaming)
- Pull container images from ECR
- Access model artifacts from S3

The cell checks if the role already exists before creating it, so the notebook is **idempotent** — safe to re-run.

In [ ]:
import json

ROLE_NAME = "SageMakerVLLMEndpointDeployRole"

# Trust policy: allow SageMaker service to assume this role
TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
})

# Inline policy: SageMaker endpoint lifecycle + invoke permissions
ENDPOINT_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "SageMakerEndpointManagement",
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateModel", "sagemaker:DeleteModel", "sagemaker:DescribeModel",
                "sagemaker:CreateEndpointConfig", "sagemaker:DeleteEndpointConfig", "sagemaker:DescribeEndpointConfig",
                "sagemaker:CreateEndpoint", "sagemaker:DeleteEndpoint", "sagemaker:DescribeEndpoint",
                "sagemaker:UpdateEndpoint", "sagemaker:InvokeEndpoint", "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        },
        {"Sid": "ECRAccess", "Effect": "Allow", "Action": ["ecr:GetAuthorizationToken", "ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer", "ecr:BatchCheckLayerAvailability"], "Resource": "*"},
        {"Sid": "S3ModelAccess", "Effect": "Allow", "Action": ["s3:GetObject", "s3:ListBucket"], "Resource": "*"},
        {"Sid": "CloudWatchLogging", "Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents", "logs:DescribeLogStreams"], "Resource": "arn:aws:logs:*:*:log-group:/aws/sagemaker/*"}
    ]
})

# Check if role already exists
try:
    role_response = iam.get_role(RoleName=ROLE_NAME)
    role_arn = role_response["Role"]["Arn"]
    print(f"✓ IAM role '{ROLE_NAME}' already exists.")
    print(f"  ARN: {role_arn}")
except iam.exceptions.NoSuchEntityException:
    print(f"Creating IAM role '{ROLE_NAME}'...")
    create_response = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=TRUST_POLICY, Description="IAM role for deploying and invoking SageMaker vLLM endpoints")
    role_arn = create_response["Role"]["Arn"]
    iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="SageMakerVLLMEndpointPolicy", PolicyDocument=ENDPOINT_POLICY)
    print(f"✓ IAM role created successfully. ARN: {role_arn}")
    print("⏳ Waiting 10s for IAM role propagation...")
    time.sleep(10)

print(f"\nRole ARN to use: {role_arn}")

## Section 3: Deploy Qwen3.5-9B to SageMaker

We deploy using the **vLLM Server DLC v2.0** (vLLM 0.22.1, CUDA 13.0.2, released 2026-06-05) which provides:
- OpenAI-compatible API format
- Tool calling support (`qwen3_coder` parser)
- Reasoning/thinking mode support (`qwen3` parser)

**Instance**: `ml.g6e.2xlarge` (1x NVIDIA L40S GPU, 48GB VRAM) — sufficient for the 9B model in BF16.

> **⚠️ Service Quota Requirement:** Ensure your AWS region has an approved instance limit for `ml.g6e.2xlarge` for SageMaker endpoint usage.

In [ ]:
# Model and instance configuration
model_id = "Qwen/Qwen3.5-9B"
instance_type = "ml.g6e.2xlarge"  # 1x L40S (48GB VRAM)
num_gpu = 1

# Resource naming
model_name = f"qwen35-9b-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 1200

# Persist to .env for cross-notebook reference
if not ENV_FILE.exists():
    ENV_FILE.touch()
set_key(str(ENV_FILE), "SAGEMAKER_ENDPOINT_NAME", endpoint_name)
set_key(str(ENV_FILE), "SAGEMAKER_MODEL_NAME", model_name)

print(f"Model ID:      {model_id}")
print(f"Instance:      {instance_type}")
print(f"Endpoint Name: {endpoint_name}")

In [ ]:
# vLLM DLC container image — vLLM 0.22.1, Python 3.12, CUDA 13.0, Ubuntu 22.04
inference_image = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/vllm:0.22.1-gpu-py312-cu130-ubuntu22.04-sagemaker"

env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(num_gpu),
    "SM_VLLM_MAX_MODEL_LEN": "32768",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "qwen3_coder",
    "SM_VLLM_REASONING_PARSER": "qwen3"
}

# Required for GPU deployments with CUDA 13 images (driver compatibility)
inference_ami_version = "al2023-ami-sagemaker-inference-gpu-4-1"

print("Container:", inference_image)
print("AMI Version:", inference_ami_version)
print("\nEnvironment variables:")
for k, v in env.items():
    print(f"  {k} = {v}")

In [ ]:
# Create SageMaker Model
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role_arn,
    PrimaryContainer={"Image": inference_image, "Environment": env},
)
print(f"✓ Model '{model_name}' created")

In [ ]:
# Create Endpoint Config and Endpoint
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        "VariantName": "v1", "ModelName": model_name,
        "InstanceType": instance_type, "InitialInstanceCount": 1,
        "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        "InferenceAmiVersion": inference_ami_version,
    }],
)
print(f"✓ Endpoint config '{endpoint_config_name}' created")

_ = sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
print(f"✓ Endpoint '{endpoint_name}' creation initiated")
print("\n⏳ This typically takes 8-15 minutes...\n")

wait_for_endpoint(endpoint_name)

---
## Section 4: Test the Endpoint

In [ ]:
# Test: Non-Streaming Text Inference
payload = {
    "messages": [{"role": "user", "content": "What are 3 interesting facts about the Python programming language?"}],
    "max_tokens": 4096, "temperature": 0.7, "top_p": 0.8,
    "chat_template_kwargs": {"enable_thinking": False},
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name, Body=json.dumps(payload), ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

message = response["choices"][0]["message"]
print(f"✅ Response time: {end_time - start_time:.2f}s\n")
print("--- Response ---")
print(message.get("content") or "[No content]")

usage = response.get("usage", {})
print(f"\n--- Usage ---")
print(f"Prompt tokens: {usage.get('prompt_tokens', 'N/A')}")
print(f"Completion tokens: {usage.get('completion_tokens', 'N/A')}")

In [ ]:
# Test: Streaming Inference
streaming_payload = {
    "messages": [{"role": "user", "content": "Write a haiku about cloud computing."}],
    "max_tokens": 4096, "temperature": 0.8, "stream": True,
    "chat_template_kwargs": {"enable_thinking": False},
}

response = sm_runtime.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name, ContentType='application/json',
    Accept='application/json', Body=json.dumps(streaming_payload),
)

print("Streaming response:")
print("=" * 60)
for event in response['Body']:
    chunk = event['PayloadPart']['Bytes'].decode('utf-8')
    for line in chunk.split('\n'):
        if line.startswith('data: '):
            json_str = line[6:].strip()
            if json_str and json_str != '[DONE]':
                try:
                    chunk_data = json.loads(json_str)
                    if 'choices' in chunk_data and chunk_data['choices']:
                        text = chunk_data['choices'][0].get('delta', {}).get('content') or ''
                        if text:
                            print(text, end='', flush=True)
                except json.JSONDecodeError:
                    continue
print("\n" + "=" * 60)
print("\n✅ Streaming test passed!")

### Test: Default Strands Agent

In [ ]:
from strands.agent import Agent
from strands.models.sagemaker import SageMakerAIModel

provider = SageMakerAIModel(
    endpoint_config={"endpoint_name": endpoint_name, "region_name": REGION},
    payload_config={"max_tokens": 1000, "temperature": 0.7, "stream": True}
)

agent = Agent(name="qwen-assistant", model=provider, system_prompt="You are a helpful AI assistant.")
print(f"Agent response: \n")
response = agent("Hello! What model are you?")
print(f"Agent: {response}")

---
## Section 5: Invoke the Endpoint Using the OpenAI-Compatible API

SageMaker AI endpoints now expose an `/openai/v1` path that accepts Chat Completions requests.

**Base URL format:**
```
https://runtime.sagemaker.<REGION>.amazonaws.com/endpoints/<ENDPOINT_NAME>/openai/v1
```

In [ ]:
import httpx
from openai import OpenAI
from sagemaker.core.token_generator import generate_token


class SageMakerAuth(httpx.Auth):
    """Auto-refreshing auth that generates a fresh bearer token on each request."""
    def __init__(self, region: str):
        self.region = region
    def auth_flow(self, request):
        request.headers["Authorization"] = f"Bearer {generate_token(region=self.region)}"
        yield request


base_url = f"https://runtime.sagemaker.{REGION}.amazonaws.com/endpoints/{endpoint_name}/openai/v1"
http_client = httpx.Client(auth=SageMakerAuth(region=REGION))

client_auto = OpenAI(base_url=base_url, api_key="sagemaker", http_client=http_client)

response = client_auto.chat.completions.create(
    model="",
    messages=[{"role": "user", "content": "Say hello in three languages."}],
    max_tokens=2048,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print("Auto-refresh token response:")
print(response.choices[0].message.content)
print("\n✅ Auto-refreshing token pattern works!")

### Use with Strands Agents via OpenAI-Compatible API

In [ ]:
from openai import AsyncOpenAI
from strands import Agent
from strands.models.openai import OpenAIModel
from sagemaker.core.token_generator import generate_token
import openai as _openai

base_url = f"https://runtime.sagemaker.{REGION}.amazonaws.com/endpoints/{endpoint_name}/openai/v1"

strands_client = AsyncOpenAI(base_url=base_url, api_key=generate_token(region=REGION))

openai_model = OpenAIModel(
    client=strands_client, model_id="",
    params={"temperature": 0.7, "max_tokens": 4096, "tools": _openai.NOT_GIVEN,
            "extra_body": {"chat_template_kwargs": {"enable_thinking": False}}},
)

openai_agent = Agent(
    model=openai_model,
    system_prompt="You are a helpful AI assistant running on Amazon SageMaker via the OpenAI-compatible API.",
)

result = openai_agent("What's the capital of France and what is it famous for? Answer in 2-3 sentences.")
print(f"Agent response: {result}")
print("\n✅ Strands Agent working via OpenAI-compatible API!")